# Hyper-Adaptive Momentum Dynamics for Native Cubic Portfolio Optimization

## Strategy Description
This notebook implements the strategy from the paper "Hyper-Adaptive Momentum Dynamics for Native Cubic Portfolio Optimization: Avoiding Quadratization Distortion in Higher-Order Cardinality-Constrained Search" by Greg Serbarinov. The strategy focuses on cubic cardinality-constrained portfolio optimization, using a hybrid pipeline combining continuous Hamiltonian search, exact cardinality-preserving projection, and iterated local search (ILS).

**Paper Citation:**
Serbarinov, G. (2026). Hyper-Adaptive Momentum Dynamics for Native Cubic Portfolio Optimization: Avoiding Quadratization Distortion in Higher-Order Cardinality-Constrained Search. *arXiv preprint arXiv:2603.15947*.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

In this phase, we set up the configuration for our trading strategy. This includes defining the universe of tickers, parameters, and the hypothesis we are testing.

In [ ]:
# Configuration
UNIVERSE = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']
RISK_FREE_RATE = 0.02
REBALANCING_PERIOD = 'M'
MOMENTUM_WINDOW = 12


## Phase 2 — Data Download & Feature Computation

In this phase, we download the market data for our universe of tickers and compute the necessary features for our strategy.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download data
data = yf.download(UNIVERSE, start='2010-01-01', end='2023-01-01', group_by='ticker')
prices = data['Adj Close']

# Compute momentum
momentum = prices.pct_change(periods=MOMENTUM_WINDOW).dropna()


## Phase 3 — Signal Generation & Portfolio Construction

In this phase, we generate trading signals based on the computed features and construct the portfolio.

In [ ]:
# Generate signals
signals = momentum.rank(axis=1, ascending=False).apply(lambda x: pd.qcut(x, q=10, labels=False))
signals = signals.shift(1)

# Position sizing
positions = signals.apply(lambda x: np.where(x > 7, 1/(signals > 7).sum(axis=1), 0), axis=1)


## Phase 4 — Vectorized Backtest

In this phase, we perform a vectorized backtest of our strategy, ensuring no look-ahead bias by shifting signals forward by 1 period.

In [ ]:
# Calculate returns
returns = prices.pct_change().dropna()

# Calculate portfolio returns
portfolio_returns = (returns * positions).sum(axis=1)


## Phase 5 — Performance Metrics

In this phase, we calculate various performance metrics for our strategy, including Sharpe, Sortino, Calmar ratios, max drawdown, and plot the equity curve.

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import norm

# Calculate performance metrics
annual_return = portfolio_returns.mean() * 12
annual_volatility = portfolio_returns.std() * np.sqrt(12)
sharpe_ratio = (annual_return - RISK_FREE_RATE) / annual_volatility
sortino_ratio = (annual_return - RISK_FREE_RATE) / portfolio_returns[portfolio_returns < 0].std() * np.sqrt(12)
max_drawdown = (prices / prices.cummax() - 1).min()
calmar_ratio = annual_return / abs(max_drawdown)

# Plot equity curve
cumulative_returns = (1 + portfolio_returns).cumprod()
plt.plot(cumulative_returns.index, cumulative_returns)
plt.title('Equity Curve')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns')
plt.show()

print(f'Sharpe Ratio: {sharpe_ratio:.2f}')
print(f'Sortino Ratio: {sortino_ratio:.2f}')
print(f'Calmar Ratio: {calmar_ratio:.2f}')
print(f'Max Drawdown: {max_drawdown:.2%}')

## Phase 6 — Monitoring Stub

In this phase, we create a function that prints the daily P&L and current positions given live data.

In [ ]:
def monitor_positions(prices, positions):
    current_prices = prices.iloc[-1]
    current_positions = positions.iloc[-1]
    daily_pnl = (current_prices * current_positions).sum() - (prices.iloc[-2] * positions.iloc[-2]).sum()
    print(f'Daily P&L: {daily_pnl:.2f}')
    print('Current Positions:')
    print(current_positions[current_positions!= 0])

# Example usage
monitor_positions(prices, positions)